# M2 output checks (cell by cell)

Checks every file M2 writes against the raw CSVs in `data/`, recomputing each number independently instead of trusting the pipeline.

| File | Checked in |
|---|---|
| `audit_flags.parquet` | sections 2 to 9 |
| `contractor_scorecard.parquet` | section 10 |
| `ward_scorecard.parquet` | section 11 |
| `scorecard_text.json` | section 12 |

Every check prints `PASS` or `FAIL` and is recorded. No cell stops on a failure, so you can run all cells and read the summary at the end (section 13).

Run with the project venv kernel. All data is synthetic and all contractors are fictional.

## 0. Setup

In [ ]:
import json, os, re, sys
from pathlib import Path

import numpy as np
import pandas as pd

# find the repo root (the folder that has contracts/ and data/) from wherever the notebook runs
ROOT = Path.cwd()
while not ((ROOT / "contracts").is_dir() and (ROOT / "data").is_dir()):
    if ROOT.parent == ROOT:
        raise SystemExit("Could not find the repo root. Open this notebook inside Impact-lab-team-6.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
OUT = ROOT / "outputs"
pd.set_option("display.width", 200, "display.max_columns", 30, "display.max_colwidth", 80)
print("repo root:", ROOT)
print("outputs  :", OUT)

In [ ]:
RESULTS = []
SECTION = "0 setup"

def check(name, ok, detail=""):
    """Record and print one check. Never raises, so every cell keeps running."""
    ok = bool(ok)
    RESULTS.append({"section": SECTION, "check": name, "result": "PASS" if ok else "FAIL", "detail": str(detail)})
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"   ({detail})" if detail != "" else ""))

## 1. Generate (or reuse) the M2 outputs

Set `REGENERATE = True` to rebuild the files from `data/` first. With `False`, the cell uses the files already in `outputs/` and only builds them if they are missing.

In [ ]:
REGENERATE = False

M2_FILES = ["audit_flags.parquet", "contractor_scorecard.parquet", "ward_scorecard.parquet", "scorecard_text.json"]
missing = [f for f in M2_FILES if not (OUT / f).exists()]
if REGENERATE or missing:
    from pipeline.audit import run
    print("running pipeline.audit.run ... (missing:", missing, ")")
    run(DATA, OUT)

SECTION = "1 files"
for f in M2_FILES:
    p = OUT / f
    check(f"{f} exists", p.exists(), f"{p.stat().st_size:,} bytes" if p.exists() else "missing")

In [ ]:
flags = pd.read_parquet(OUT / "audit_flags.parquet")
cs = pd.read_parquet(OUT / "contractor_scorecard.parquet")
ws = pd.read_parquet(OUT / "ward_scorecard.parquet")
text = json.loads((OUT / "scorecard_text.json").read_text(encoding="utf-8"))

for name, df in [("audit_flags", flags), ("contractor_scorecard", cs), ("ward_scorecard", ws)]:
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols")
print("scorecard_text.json keys:", list(text))

### Raw data, loaded independently of the pipeline (used to recompute everything below)

In [ ]:
def read(n):
    return pd.read_csv(DATA / f"{n}.csv", encoding="utf-8")

wo_raw, rc, roads, wards, contractors, complaints = (read(n) for n in
    ["work_orders", "rate_card", "roads", "wards", "contractors", "complaints"])

wo = wo_raw.merge(rc, on="repair_method", how="left").merge(roads[["road_id", "under_dlp"]], on="road_id", how="left")
for col in ("assigned_at", "completed_at"):
    wo[col] = pd.to_datetime(wo[col])
wo["expected"] = np.where(wo.unit == "per sqm", wo.area_sqm * wo.standard_rate_inr, wo.standard_rate_inr)
wo["ratio"] = wo.cost_inr / wo.expected
wo["unlinked"] = wo.linked_complaint_ids.fillna("").str.strip().eq("")
W = wo.set_index("work_order_id")
print(len(wo), "work orders,", wo.expected.isna().sum(), "without a rate-card match")

## 2. Contract (columns and types from `contracts/schemas.py`)

In [ ]:
SECTION = "2 contract"
from contracts.schemas import SCHEMAS, FLAG_TYPES, SEVERITIES, validate

for name, df in [("audit_flags", flags), ("contractor_scorecard", cs), ("ward_scorecard", ws)]:
    problems = validate(df, name)
    check(f"{name} matches contract", problems == [], "; ".join(problems))
    extra = sorted(set(df.columns) - set(SCHEMAS[name]))
    check(f"{name} has no extra columns", not extra, extra or "")

## 3. `audit_flags`: basic integrity

In [ ]:
SECTION = "3 flags integrity"
check("flag_id is unique", flags.flag_id.is_unique)
check("flag_type values valid", set(flags.flag_type) <= set(FLAG_TYPES))
check("severity values valid", set(flags.severity) <= set(SEVERITIES))
check("flag_id prefix matches flag_type",
      (flags.flag_id.str.extract(r"^F-([A-Z_]+)-\d{4}$")[0] == flags.flag_type).all())
check("every work_order_id exists in work_orders.csv", flags.work_order_id.isin(wo.work_order_id).all())
rel = flags.related_work_order_id.dropna()
check("every related_work_order_id exists", rel.isin(wo.work_order_id).all(), f"{len(rel)} non-null")
check("amount_at_risk_inr >= 0", (flags.amount_at_risk_inr >= 0).all())
check("no duplicate (flag_type, work_order_id)", not flags.duplicated(["flag_type", "work_order_id"]).any())

# ward / road / contractor / billed must copy the work order exactly
j = flags.join(W[["ward_code", "road_id", "contractor_id", "cost_inr"]], on="work_order_id", rsuffix="_wo")
for col in ("ward_code", "road_id", "contractor_id"):
    check(f"{col} matches the work order", (j[col] == j[f"{col}_wo"]).all())
check("billed_inr equals work order cost", np.allclose(j.billed_inr, j.cost_inr))

print("\nnulls per column:")
print(flags.isna().sum().to_string())

In [ ]:
SECTION = "3 flags integrity"
EXPECTED = {"OVERBILL": 114, "DOUBLE_BILL": 45, "DLP_PAID": 493, "SLA_MISS": 1259, "REPEAT_REPAIR": 611}
got = flags.flag_type.value_counts()
for t, n in EXPECTED.items():
    check(f"{t} count = {n}", got.get(t, 0) == n, f"got {got.get(t, 0)}")
flags.groupby(["flag_type", "severity"]).size().unstack(fill_value=0)

## 4. OVERBILL, recomputed from the rate card

In [ ]:
SECTION = "4 OVERBILL"
f = flags[flags.flag_type == "OVERBILL"].set_index("work_order_id")
truth = set(wo.loc[wo.ratio > 1.30, "work_order_id"])
check("same work orders as ratio > 1.30", set(f.index) == truth,
      f"missing {len(truth - set(f.index))}, extra {len(set(f.index) - truth)}")
check("no job inside the tolerance band is flagged", (W.loc[f.index, "ratio"] > 1.30).all())
check("expected_inr = rate card expected", np.allclose(f.expected_inr, W.loc[f.index, "expected"]))
risk = (W.loc[f.index, "cost_inr"] - 1.30 * W.loc[f.index, "expected"]).clip(lower=0).round(0)
check("amount_at_risk = billed - 1.30 x expected", np.allclose(f.amount_at_risk_inr, risk))
sev = np.where(W.loc[f.index, "ratio"] > 1.50, "high", "medium")
check("severity high above 1.50x, else medium", (f.severity.to_numpy() == sev).all())
print(f"total excess: ₹{f.amount_at_risk_inr.sum():,.0f}  (blueprint ₹44.8 L)")
f[["severity", "billed_inr", "expected_inr", "amount_at_risk_inr", "evidence"]].head()

## 5. DOUBLE_BILL, recomputed (no complaint linked + earlier same road and work type job within 7 days)

In [ ]:
SECTION = "5 DOUBLE_BILL"
f = flags[flags.flag_type == "DOUBLE_BILL"].set_index("work_order_id")

# brute force: for every unlinked WO, look for ANY earlier same road + work type WO within 7 days
truth = {}
for r in wo[wo.unlinked].itertuples():
    prior = wo[(wo.road_id == r.road_id) & (wo.work_type == r.work_type) & (wo.work_order_id != r.work_order_id)
               & (wo.assigned_at <= r.assigned_at) & (wo.assigned_at >= r.assigned_at - pd.Timedelta(days=7))]
    if len(prior):
        truth[r.work_order_id] = set(prior.work_order_id)

check("same work orders as the brute-force rule", set(f.index) == set(truth),
      f"missing {sorted(set(truth) - set(f.index))[:5]}, extra {sorted(set(f.index) - set(truth))[:5]}")
check("every flag has an earlier job", f.related_work_order_id.notna().all())
check("earlier job is one of the matching prior jobs",
      all(f.loc[k, "related_work_order_id"] in truth.get(k, ()) for k in f.index))
check("no double-bill flag on a job that has a linked complaint", W.loc[f.index, "unlinked"].all())
check("amount_at_risk = full cost", np.allclose(f.amount_at_risk_inr, W.loc[f.index, "cost_inr"]))
check("expected_inr = 0", (f.expected_inr == 0).all())
print(f"total: ₹{f.amount_at_risk_inr.sum():,.0f}  (blueprint ₹25.9 L)")
f[["related_work_order_id", "billed_inr", "evidence"]].head()

## 6. DLP_PAID, recomputed from `roads.under_dlp`

In [ ]:
SECTION = "6 DLP_PAID"
f = flags[flags.flag_type == "DLP_PAID"].set_index("work_order_id")
truth = set(wo.loc[(wo.under_dlp == "Y") & (wo.cost_inr > 0), "work_order_id"])
check("same work orders as under_dlp = Y and cost > 0", set(f.index) == truth,
      f"missing {len(truth - set(f.index))}, extra {len(set(f.index) - truth)}")
check("amount_at_risk = full cost", np.allclose(f.amount_at_risk_inr, W.loc[f.index, "cost_inr"]))
check("roads covered = 61", f.road_id.nunique() == 61, f"got {f.road_id.nunique()}")
print(f"total recoverable: ₹{f.amount_at_risk_inr.sum():,.0f}  (blueprint ₹2.62 Cr)")

## 7. SLA_MISS, recomputed (hours from the first linked complaint)

In [ ]:
SECTION = "7 SLA_MISS"
f = flags[flags.flag_type == "SLA_MISS"].set_index("work_order_id")
truth = set(wo.loc[wo.sla_met == "N", "work_order_id"])
check("same work orders as sla_met = N", set(f.index) == truth,
      f"missing {len(truth - set(f.index))}, extra {len(set(f.index) - truth)}")
check("amount_at_risk = 0 (deadline flag, no money)", (f.amount_at_risk_inr == 0).all())

# recompute hours late from complaints.csv
c_time = pd.to_datetime(complaints.set_index("complaint_id").received_at)
first = (wo.dropna(subset=["linked_complaint_ids"]).set_index("work_order_id").linked_complaint_ids
         .str.split("|").explode().str.strip().map(c_time).groupby(level=0).min())
late = ((W.completed_at - first.reindex(W.index)).dt.total_seconds() / 3600 - W.sla_hours).round(1)
shown = f.evidence.str.extract(r"\(([\d.]+) h late\)")[0].astype(float).dropna()
check("hours late in evidence matches recompute", np.allclose(shown, late.loc[shown.index]),
      f"{len(shown)} with a complaint time")
not_late = late.loc[shown.index][lambda s: s <= 0]
check("every SLA_MISS is actually late by the recompute", not_late.empty,
      f"{len(not_late)} not late: {not_late.to_dict()}")
f[["severity", "evidence"]].head()

**Known result for the check above:** it fails on 1 job, `WO25-00541`. That job finished exactly at its 24 h deadline (0 h late), but the source data marks it `sla_met = N`. The blueprint (§2, A4) calls this the 1 boundary case out of 2,499 and decides to trust the `sla_met` column, so M2 still flags it. The failure is left visible on purpose rather than hidden.

In [ ]:
SECTION = "7 SLA_MISS"
print(f.loc[["WO25-00541"], ["severity", "evidence"]].to_string())

## 8. REPEAT_REPAIR, recomputed (follow-up on the same road within 30 days of completion)

In [ ]:
SECTION = "8 REPEAT_REPAIR"
f = flags[flags.flag_type == "REPEAT_REPAIR"].set_index("work_order_id")
truth = set(wo.loc[wo.reopened_within_30d == "Y", "work_order_id"])
check("same work orders as reopened_within_30d = Y", set(f.index) == truth,
      f"missing {len(truth - set(f.index))}, extra {len(set(f.index) - truth)}")

bad = []
for k, r in f.iterrows():
    base = W.loc[k]
    cand = wo[(wo.road_id == base.road_id) & (wo.work_order_id != k) & (wo.assigned_at >= base.completed_at)
              & (wo.assigned_at <= base.completed_at + pd.Timedelta(days=30))].sort_values(["assigned_at", "work_order_id"])
    exp_id = cand.work_order_id.iloc[0] if len(cand) else None
    exp_cost = float(cand.cost_inr.iloc[0]) if len(cand) else 0.0
    got_id = None if pd.isna(r.related_work_order_id) else r.related_work_order_id   # parquet stores "none" as NaN
    if got_id != exp_id or not np.isclose(r.amount_at_risk_inr, exp_cost):
        bad.append(k)
check("follow-up WO is the first same-road job within 30 days, ₹ = its cost", not bad, f"{len(bad)} mismatches {bad[:5]}")
share = f.related_work_order_id.notna().mean()
check("about 82% have a follow-up (blueprint)", 0.80 <= share <= 0.84, f"{share:.1%}")
print(f"paid-twice total: ₹{f.amount_at_risk_inr.sum():,.0f}")

## 9. Evidence and explanation text

In [ ]:
SECTION = "9 flag text"
BANNED = re.compile(r"\b(?:fraud\w*|corrupt\w*|scam\w*|theft|thief|thieves|stole|steal\w*|loot\w*|"
                    r"embezzl\w*|kickback\w*|cheat\w*|swindl\w*|ghotala|bhrashtachar)\b", re.I)
ID = re.compile(r"\bWO25-\d{5}\b")
DASH = "[–—]"

for col in ("evidence", "explanation"):
    s = flags[col].fillna("")
    check(f"{col}: never empty", s.str.len().gt(0).all(), f"{s.str.len().eq(0).sum()} empty")
    check(f"{col}: no accusing words", not s.str.contains(BANNED).any(), s[s.str.contains(BANNED)].head(3).tolist())
    check(f"{col}: no em or en dashes", not s.str.contains(DASH).any())

words = flags.explanation.str.split().str.len()
check("explanation <= 25 words", words.le(25).all(), f"max {words.max()}")
check("explanation names its own work order", flags.apply(lambda r: r.work_order_id in r.explanation, axis=1).all())
own = flags.apply(lambda r: set(ID.findall(r.explanation)) <= {r.work_order_id, r.related_work_order_id}, axis=1)
check("explanation cites no unrelated work order", own.all(), f"{(~own).sum()} bad")
flags.groupby("flag_type").head(1)[["flag_id", "evidence", "explanation"]]

## 10. `contractor_scorecard`, recomputed

In [ ]:
SECTION = "10 contractor scorecard"
c = cs.set_index("contractor_id")
g = wo.groupby("contractor_id")
def pct_y(col):
    return (100 * g[col].apply(lambda s: (s == "Y").mean())).round(1).reindex(c.index)

check("8 contractors, unique", len(cs) == 8 and cs.contractor_id.is_unique)
check("names match contractors.csv",
      (c.contractor_name == contractors.set_index("contractor_id").contractor_name.reindex(c.index)).all())
check("n_jobs", (c.n_jobs == g.size().reindex(c.index)).all())
check("billed_inr", np.allclose(c.billed_inr, g.cost_inr.sum().reindex(c.index)))
check("sla_pct", np.allclose(c.sla_pct, pct_y("sla_met")))
check("reopen_pct", np.allclose(c.reopen_pct, pct_y("reopened_within_30d")))
check("median_cost_ratio", np.allclose(c.median_cost_ratio, g.ratio.median().round(3).reindex(c.index)))

# method-mix expected re-open rate
mrate = wo.groupby("repair_method").reopened_within_30d.apply(lambda s: (s == "Y").mean())
exp = wo.assign(e=wo.repair_method.map(mrate)).groupby("contractor_id").e.mean()
check("reopen_expected_pct (method-mix adjusted)", np.allclose(c.reopen_expected_pct, (100 * exp).round(1).reindex(c.index)))
check("durability_index = expected / actual",
      np.allclose(c.durability_index, c.reopen_expected_pct / c.reopen_pct, atol=0.02))

In [ ]:
SECTION = "10 contractor scorecard"
counts = flags.groupby(["contractor_id", "flag_type"]).size().unstack(fill_value=0)
ok = all(json.loads(c.loc[k, "n_flags_by_type"]) == {t: int(counts.loc[k].get(t, 0)) for t in FLAG_TYPES} for k in c.index)
check("n_flags_by_type JSON matches the flags", ok)

# money is keyed to the job that was actually billed: a repeat repair's amount is the follow-up job's bill
mf = flags[flags.flag_type != "SLA_MISS"]
billed_job = mf.related_work_order_id.where((mf.flag_type == "REPEAT_REPAIR") & mf.related_work_order_id.notna(), mf.work_order_id)
money = mf.groupby(billed_job).amount_at_risk_inr.max()
amt = money.rename("m").to_frame().join(W.contractor_id).groupby("contractor_id").m.sum()
check("amount_flagged_inr (largest flag per job)", np.allclose(c.amount_flagged_inr, amt.reindex(c.index).fillna(0)))

check("score between 0 and 100", c.score_0_100.between(0, 100).all())
weights, comps = text["score_weights"], text["score_components"]
recalc = pd.Series({k: round(100 * sum(weights[n] * v for n, v in comps[k].items()), 1) for k in c.index})
check("score = weighted components", np.allclose(c.score_0_100, recalc))
CUTS = [(80, "A"), (65, "B"), (50, "C"), (35, "D"), (0, "E")]
grade = c.score_0_100.apply(lambda s: next(gr for cut, gr in CUTS if s >= cut))
check("grade matches cutoffs", (grade == c.grade).all())
deccan = cs[cs.contractor_name.str.contains("Deccan")].score_0_100.iloc[0]
western = cs[cs.contractor_name.str.contains("Western Coast")].score_0_100.iloc[0]
check("Deccan outranks Western Coast", deccan > western, f"{deccan} vs {western}")
r = cs.score_0_100.corr(cs.n_jobs)
check("score not driven by volume (|corr with n_jobs| < 0.7)", abs(r) < 0.7, f"corr {r:.2f}")
cs.drop(columns="summary_text")

## 11. `ward_scorecard`, recomputed

In [ ]:
SECTION = "11 ward scorecard"
w = ws.set_index("ward_code")
wd = wards.set_index("ward_code")
gw = wo.groupby("ward_code")
def wpct_y(col):
    return (100 * gw[col].apply(lambda s: (s == "Y").mean())).round(1).reindex(w.index)

check("24 wards, unique", len(ws) == 24 and ws.ward_code.is_unique)
check("same wards as wards.csv", set(w.index) == set(wd.index))
check("budget_inr = wards.csv budget", np.allclose(w.budget_inr, wd.monsoon_repair_budget_inr_2025.reindex(w.index)))
check("billed_inr per ward", np.allclose(w.billed_inr, gw.cost_inr.sum().reindex(w.index)))
check("ward billed adds up to ₹13,43,28,183", np.isclose(ws.billed_inr.sum(), 134_328_183), f"₹{ws.billed_inr.sum():,.0f}")
check("budget_used_pct", np.allclose(w.budget_used_pct, (100 * w.billed_inr / w.budget_inr).round(1)))
famt = money.rename("m").to_frame().join(W.ward_code).groupby("ward_code").m.sum()
check("flagged_inr (largest flag per job)", np.allclose(w.flagged_inr, famt.reindex(w.index).fillna(0)))
check("flagged_pct", np.allclose(w.flagged_pct, (100 * w.flagged_inr / w.billed_inr).round(1)))
check("flagged_inr <= billed_inr", (w.flagged_inr <= w.billed_inr).all())
check("reopen_pct", np.allclose(w.reopen_pct, wpct_y("reopened_within_30d")))
check("sla_pct", np.allclose(w.sla_pct, wpct_y("sla_met")))
top = wo.groupby(["ward_code", "contractor_id"]).cost_inr.sum().groupby(level=0).idxmax().str[1]
check("top_contractor_id = most spend in the ward", (w.top_contractor_id == top.reindex(w.index)).all())
print("over budget:", (w.budget_used_pct > 100).sum(), "wards")
ws.drop(columns="summary_text").sort_values("flagged_pct", ascending=False)

## 12. `scorecard_text.json`

In [ ]:
SECTION = "12 scorecard_text.json"
need = {"contractors", "wards", "fairness_notes", "score_weights", "score_components", "thresholds", "facts"}
check("has all expected keys", need <= set(text), need - set(text) or "")
check("8 contractor texts", set(text["contractors"]) == set(cs.contractor_id))
check("24 ward texts", set(text["wards"]) == set(ws.ward_code))
check("contractor text = parquet summary_text", all(text["contractors"][k] == c.loc[k, "summary_text"] for k in c.index))
check("ward text = parquet summary_text", all(text["wards"][k] == w.loc[k, "summary_text"] for k in w.index))
check("score weights sum to 1", np.isclose(sum(text["score_weights"].values()), 1.0), text["score_weights"])
check("thresholds as in the blueprint", text["thresholds"] == {"overbill_ratio": 1.3, "overbill_high_ratio": 1.5,
      "double_bill_days": 7, "repeat_window_days": 30}, text["thresholds"])
check("fairness notes present (5 or more)", len(text["fairness_notes"]) >= 5, len(text["fairness_notes"]))

for kind, d, limit in [("contractor", text["contractors"], 120), ("ward", text["wards"], 150)]:
    s = pd.Series(d)
    n = s.str.split().str.len()
    check(f"{kind} texts <= {limit} words", n.le(limit).all(), f"max {n.max()}")
    check(f"{kind} texts: no accusing words", not s.str.contains(BANNED).any())
    check(f"{kind} texts: no em or en dashes", not s.str.contains(DASH).any())
    check(f"{kind} texts: never empty", s.str.len().gt(0).all())

In [ ]:
SECTION = "12 scorecard_text.json"
# every number in a summary must appear in its facts packet (same rule as the eval grader)
from evals.graders import numbers_in, _walk_numbers, _grounded
bad = []
for kind, texts in [("contractors", text["contractors"]), ("wards", text["wards"])]:
    for k, t in texts.items():
        allowed = set()
        _walk_numbers(text["facts"][kind][k], allowed)
        loose = [raw for raw, x in numbers_in(t) if not _grounded(x, allowed)]
        if loose:
            bad.append((k, loose))
check("every number in the summaries comes from the facts", not bad, bad[:3] or "")
print(text["contractors"][cs.contractor_id.iloc[0]], "\n")
print(text["wards"]["A"])

## 13. Summary

In [ ]:
res = pd.DataFrame(RESULTS)
print(f"{(res.result == 'PASS').sum()} PASS, {(res.result == 'FAIL').sum()} FAIL out of {len(res)} checks")
display(res.groupby("section").result.value_counts().unstack(fill_value=0))
fails = res[res.result == "FAIL"]
fails if len(fails) else print("No failures.")